# 分子对接工作流 (AutoDock Vina)

**主要功能：**
- 使用 AutoDock Vina 进行分子对接
- 支持多种配体输入格式 (SMILES, SDF, PDBQT)
- 批量对接多个蛋白质-配体组合
- 对接结果分析和可视化

**输入：**
- 蛋白质PDB文件 (需要预先进行口袋检测)
- 配体文件 (SMILES, SDF, PDBQT) 或配体列表
- 口袋坐标 (来自口袋检测工作流)

**输出：**
- 对接结果 (结合亲和力, 坐标, 构象)
- 最佳配体结合模式
- 对接结果分析报告

**系统要求：**
- AutoDock Vina 已安装
- OpenBabel (用于格式转换)
- 约 5-10 GB 磁盘空间

## 1. 环境设置与依赖安装

In [ ]:
# 使用共享工具初始化环境（自动安装依赖、设置路径）
import sys
import os
from pathlib import Path
import subprocess
import shutil

# 添加protflow到路径
project_root = Path.cwd()
while not (project_root / 'src' / 'protflow').exists() and project_root != project_root.parent:
    project_root = project_root.parent

if (project_root / 'src').exists():
    src_dir = str(project_root / 'src')
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print(f"✓ protflow 路径: {src_dir}")

# 导入并设置环境
from protflow.utils.notebook_utils import setup_notebook_environment

# 设置环境
paths = setup_notebook_environment(work_dir_name='docking_runs')

PROJECT_ROOT = paths['PROJECT_ROOT']
WORK_DIR = paths['WORK_DIR']
DATA_DIR = paths['DATA_DIR']

print(f"\n✓ 环境初始化完成")
print(f"  工作目录: {WORK_DIR}")
print(f"  项目根目录: {PROJECT_ROOT}")


In [ ]:
# 使用后端模块检查依赖（所有业务逻辑在后端）
from protflow.docking.vina_dock import check_vina_available
import shutil

# 检查依赖
vina_available = check_vina_available()
obabel_available = shutil.which('obabel') is not None

print("=== 依赖检查 ===")
print(f"AutoDock Vina: {'✓ 可用' if vina_available else '❌ 未找到'}")
print(f"OpenBabel: {'✓ 可用' if obabel_available else '❌ 未找到'}")

if not vina_available:
    print("\n⚠️ 请安装 AutoDock Vina:")
    print("  conda install -c conda-forge autodock-vina")
    print("  或访问: https://vina.scripps.edu/downloads/")


## 2. 安装AutoDock Vina (如需要)

In [ ]:
# Vina安装说明（请手动安装）
print("请手动安装 AutoDock Vina:")
print("  方式1: conda install -c conda-forge autodock-vina")
print("  方式2: 访问 https://vina.scripps.edu/downloads/")
print("\n安装后重新运行依赖检查cell")


## 3. 输入准备

### 3.1 蛋白质结构文件

In [ ]:
from pathlib import Path
import pandas as pd

def prepare_protein_files(protein_input):
    """准备蛋白质文件"""
    protein_files = []
    
    if isinstance(protein_input, str):
        protein_input = Path(protein_input)
    
    if protein_input.is_file() and protein_input.suffix == '.pdb':
        # 单个PDB文件
        protein_files.append({
            'name': protein_input.stem,
            'file': str(protein_input),
            'prepared_file': None
        })
    
    elif protein_input.is_dir():
        # 目录中的PDB文件
        pdb_files = list(protein_input.glob('*.pdb'))
        for pdb_file in pdb_files:
            protein_files.append({
                'name': pdb_file.stem,
                'file': str(pdb_file),
                'prepared_file': None
            })
    
    print(f"✓ 找到 {len(protein_files)} 个蛋白质文件")
    return protein_files

# 蛋白质输入路径
protein_input_path = "path/to/protein/files"  # 替换为您的路径
protein_files = prepare_protein_files(protein_input_path)

### 3.2 口袋坐标 (来自口袋检测)

In [ ]:
def load_docking_sites(sites_file):
    """加载对接位点坐标"""
    sites_file = Path(sites_file)
    
    if not sites_file.exists():
        print(f"❌ 位点文件不存在: {sites_file}")
        return pd.DataFrame()
    
    try:
        sites_df = pd.read_csv(sites_file)
        print(f"✓ 加载了 {len(sites_df)} 个对接位点")
        print(f"涉及 {sites_df['protein_name'].nunique()} 个蛋白质")
        
        # 显示前几行
        print("\n前5个位点:")
        print(sites_df.head())
        
        return sites_df
    
    except Exception as e:
        print(f"❌ 加载失败: {e}")
        return pd.DataFrame()

# 加载对接位点 (来自口袋检测工作流)
sites_file = "path/to/docking_sites.csv"  # 替换为实际路径
docking_sites = load_docking_sites(sites_file)

### 3.3 配体准备

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

def prepare_ligand(ligand_input, output_dir):
    """准备配体文件"""
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)
    
    ligand_info = {
        'name': None,
        'input_file': None,
        'pdbqt_file': None,
        'smiles': None,
        'status': 'failed'
    }
    
    try:
        # 处理不同类型的输入
        if ligand_input.endswith(('.smi', '.smiles')) or 'SMILES:' in ligand_input:
            # SMILES输入
            if 'SMILES:' in ligand_input:
                smiles = ligand_input.replace('SMILES:', '').strip()
            else:
                # 从文件读取SMILES
                with open(ligand_input, 'r') as f:
                    smiles = f.read().strip().split()[0]
            
            ligand_info['smiles'] = smiles
            ligand_info['name'] = 'ligand_from_smiles'
            
            # 从SMILES生成分子
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                print(f"❌ 无效的SMILES: {smiles}")
                return ligand_info
            
            # 添加氢原子
            mol = Chem.AddHs(mol)
            
            # 生成3D构象
            AllChem.EmbedMolecule(mol)
            AllChem.UFFOptimizeMolecule(mol)
            
            # 保存为SDF
            sdf_file = output_path / 'ligand.sdf'
            writer = Chem.SDWriter(str(sdf_file))
            writer.write(mol)
            writer.close()
            
            ligand_info['input_file'] = str(sdf_file)
            
        elif ligand_input.endswith(('.sdf', '.mol', '.mol2')):
            # 分子文件输入
            ligand_info['input_file'] = ligand_input
            ligand_info['name'] = Path(ligand_input).stem
            
        elif ligand_input.endswith('.pdbqt'):
            # 已经是PDBQT格式
            ligand_info['input_file'] = ligand_input
            ligand_info['pdbqt_file'] = ligand_input
            ligand_info['name'] = Path(ligand_input).stem
            ligand_info['status'] = 'ready'
            return ligand_info
        
        else:
            print(f"❌ 不支持的配体格式: {ligand_input}")
            return ligand_info
        
        # 转换为PDBQT格式
        if ligand_info['input_file']:
            pdbqt_file = output_path / f"{ligand_info['name']}.pdbqt"
            
            # 使用OpenBabel转换
            cmd = ['obabel', ligand_info['input_file'], '-O', str(pdbqt_file), '-xh']
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0 and pdbqt_file.exists():
                ligand_info['pdbqt_file'] = str(pdbqt_file)
                ligand_info['status'] = 'ready'
                print(f"✓ 配体 {ligand_info['name']} 准备完成")
            else:
                print(f"❌ PDBQT转换失败: {result.stderr}")
        
    except Exception as e:
        print(f"❌ 配体准备失败: {e}")
    
    return ligand_info

# 示例配体输入
ligand_input = "CC(=O)OC1=CC=CC=C1C(=O)O"  # 阿司匹林SMILES
# ligand_input = "path/to/ligand.sdf"  # 或者文件路径

ligand_info = prepare_ligand(ligand_input, WORK_DIR / 'ligand_prep')
print(f"配体状态: {ligand_info['status']}")

## 4. 蛋白质预处理

In [ ]:
def prepare_protein_for_docking(protein_file, output_dir):
    """预处理蛋白质文件"""
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)
    
    protein_name = Path(protein_file).stem
    pdbqt_file = output_path / f"{protein_name}.pdbqt"
    
    try:
        # 使用OpenBabel处理蛋白质
        # 1. 删除水分子
        # 2. 添加极性氢原子
        # 3. 分配部分电荷
        cmd = [
            'obabel', protein_file,
            '-O', str(pdbqt_file),
            '-xr',  # 删除水
            '-xh',  # 添加极性氢
            '-xp',  # 分配部分电荷
            '-f', 'pdb'
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0 and pdbqt_file.exists():
            print(f"✓ 蛋白质 {protein_name} 预处理完成")
            return str(pdbqt_file)
        else:
            print(f"❌ 蛋白质预处理失败: {result.stderr}")
            return None
    
    except Exception as e:
        print(f"❌ 蛋白质预处理错误: {e}")
        return None

# 预处理蛋白质文件
if protein_files:
    prepared_proteins = []
    prep_dir = WORK_DIR / 'prepared_proteins'
    
    for protein in protein_files:
        pdbqt_file = prepare_protein_for_docking(protein['file'], prep_dir)
        if pdbqt_file:
            protein['prepared_file'] = pdbqt_file
            prepared_proteins.append(protein)
    
    print(f"✓ 预处理完成: {len(prepared_proteins)}/{len(protein_files)} 个蛋白质")
    protein_files = prepared_proteins

## 5. 分子对接配置

In [ ]:
def create_vina_config(protein_pdbqt, ligand_pdbqt, center_coords, size=[20, 20, 20], exhaustiveness=8):
    """创建Vina配置文件"""
    config_content = f"""
receptor = {protein_pdbqt}
ligand = {ligand_pdbqt}

center_x = {center_coords[0]}
center_y = {center_coords[1]}
center_z = {center_coords[2]}

size_x = {size[0]}
size_y = {size[1]}
size_z = {size[2]}

exhaustiveness = {exhaustiveness}
num_modes = 9
energy_range = 3
"""
    return config_content.strip()

def run_vina_docking(config_content, output_file, log_file):
    """运行Vina对接"""
    try:
        # 创建临时配置文件
        config_file = WORK_DIR / 'vina_config.txt'
        with open(config_file, 'w') as f:
            f.write(config_content)
        
        # 运行Vina
        cmd = ['vina', '--config', str(config_file), '--out', str(output_file), '--log', str(log_file)]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)  # 30分钟超时
        
        if result.returncode == 0:
            if Path(output_file).exists():
                print(f"✓ 对接完成: {output_file.name}")
                return True
            else:
                print(f"❌ 输出文件未生成")
                return False
        else:
            print(f"❌ Vina运行失败: {result.stderr[:200]}...")
            return False
    
    except subprocess.TimeoutExpired:
        print(f"⏰ 对接超时 (30分钟)")
        return False
    
    except Exception as e:
        print(f"❌ 对接错误: {e}")
        return False

# 对接参数配置
docking_params = {
    'box_size': [20, 20, 20],  # 对接盒子大小 (Angstrom)
    'exhaustiveness': 8,       # 搜索详尽程度 (1-32)
    'num_modes': 9,            # 输出构象数量
    'energy_range': 3          # 能量范围 (kcal/mol)
}

print("对接参数:")
for key, value in docking_params.items():
    print(f"  {key}: {value}")

## 6. 批量分子对接

In [ ]:
from tqdm import tqdm
import re

def run_batch_docking(protein_files, ligand_info, docking_sites, docking_params):
    """批量运行分子对接"""
    
    if ligand_info['status'] != 'ready':
        print("❌ 配体未准备好")
        return []
    
    if not protein_files:
        print("❌ 没有蛋白质文件")
        return []
    
    if docking_sites.empty:
        print("❌ 没有对接位点")
        return []
    
    # 创建输出目录
    output_dir = WORK_DIR / 'docking_results'
    output_dir.mkdir(exist_ok=True, parents=True)
    
    results = []
    total_combinations = len(protein_files) * len(docking_sites)
    
    print(f"准备对接 {total_combinations} 个组合...")
    print(f"蛋白质: {len(protein_files)} 个")
    print(f"对接位点: {len(docking_sites)} 个")
    
    # 进度条
    with tqdm(total=total_combinations, desc="分子对接") as pbar:
        
        for protein in protein_files:
            if not protein.get('prepared_file'):
                continue
            
            # 找到该蛋白质的对接位点
            protein_sites = docking_sites[docking_sites['protein_name'] == protein['name']
            
            if protein_sites.empty:
                continue
            
            for _, site in protein_sites.iterrows():
                try:
                    # 创建配置文件
                    center_coords = [site['center_x'], site['center_y'], site['center_z']]
                    config = create_vina_config(
                        protein['prepared_file'],
                        ligand_info['pdbqt_file'],
                        center_coords,
                        docking_params['box_size'],
                        docking_params['exhaustiveness']
                    )
                    
                    # 输出文件
                    output_file = output_dir / f"{protein['name']}_site{site['pocket_rank']}_docked.pdbqt"
                    log_file = output_dir / f"{protein['name']}_site{site['pocket_rank']}_log.txt"
                    
                    # 运行对接
                    success = run_vina_docking(config, output_file, log_file)
                    
                    result = {
                        'protein_name': protein['name'],
                        'pocket_rank': site['pocket_rank'],
                        'center_x': site['center_x'],
                        'center_y': site['center_y'],
                        'center_z': site['center_z'],
                        'output_file': str(output_file) if success else None,
                        'log_file': str(log_file),
                        'status': 'success' if success else 'failed'
                    }
                    
                    results.append(result)
                    
                except Exception as e:
                    print(f"❌ 对接错误: {e}")
                    results.append({
                        'protein_name': protein['name'],
                        'pocket_rank': site['pocket_rank'],
                        'center_x': site['center_x'],
                        'center_y': site['center_y'],
                        'center_z': site['center_z'],
                        'output_file': None,
                        'log_file': None,
                        'status': f'error: {e}'
                    })
                
                pbar.update(1)
    
    print(f"✓ 批量对接完成")
    successful = len([r for r in results if r['status'] == 'success'])
    print(f"成功: {successful}/{len(results)} 个对接")
    
    return results

# 运行批量对接
if vina_ok and ligand_info['status'] == 'ready' and not docking_sites.empty:
    docking_results = run_batch_docking(protein_files, ligand_info, docking_sites, docking_params)
else:
    print("⚠️ 无法进行对接: 缺少必要条件")
    docking_results = []

## 7. 对接结果分析

In [ ]:
def parse_vina_results(results):
    """解析Vina对接结果"""
    
    for result in results:
        if result['status'] != 'success' or not result['output_file']:
            continue
        
        try:
            output_file = Path(result['output_file'])
            
            if not output_file.exists():
                continue
            
            # 读取PDBQT文件，提取能量信息
            with open(output_file, 'r') as f:
                lines = f.readlines()
            
            # 查找能量信息
            binding_energies = []
            for line in lines:
                if line.startswith('REMARK VINA RESULT:'):
                    parts = line.split()
                    if len(parts) >= 4:
                        energy = float(parts[3])
                        binding_energies.append(energy)
            
            if binding_energies:
                result['binding_energies'] = binding_energies
                result['best_energy'] = min(binding_energies)
                result['mean_energy'] = sum(binding_energies) / len(binding_energies)
                result['num_poses'] = len(binding_energies)
            
            # 解析日志文件获取更多信息
            log_file = Path(result['log_file'])
            if log_file.exists():
                with open(log_file, 'r') as f:
                    log_content = f.read()
                
                # 提取运行时间
                time_match = re.search(r'([\d.]+)\s*seconds', log_content)
                if time_match:
                    result['docking_time'] = float(time_match.group(1))
        
        except Exception as e:
            print(f"❌ 解析结果失败 {result['output_file']}: {e}")
            result['parsing_error'] = str(e)
    
    return results

def analyze_docking_results(results):
    """分析对接结果"""
    
    successful_results = [r for r in results if r['status'] == 'success' and 'best_energy' in r]
    
    if not successful_results:
        print("⚠️ 没有成功的对接结果")
        return pd.DataFrame()
    
    # 创建结果数据框
    df_data = []
    for result in successful_results:
        df_data.append({
            'protein_name': result['protein_name'],
            'pocket_rank': result['pocket_rank'],
            'center_x': result['center_x'],
            'center_y': result['center_y'],
            'center_z': result['center_z'],
            'best_energy': result['best_energy'],
            'mean_energy': result.get('mean_energy', 0),
            'num_poses': result.get('num_poses', 0),
            'docking_time': result.get('docking_time', 0),
            'output_file': result['output_file']
        })
    
    results_df = pd.DataFrame(df_data)
    
    print("=== 对接结果分析 ===")
    print(f"成功对接: {len(successful_results)} 个")
    
    if len(successful_results) > 0:
        print(f"最佳结合能: {results_df['best_energy'].min():.2f} kcal/mol")
        print(f"平均结合能: {results_df['best_energy'].mean():.2f} kcal/mol")
        print(f"结合能范围: {results_df['best_energy'].min():.2f} - {results_df['best_energy'].max():.2f} kcal/mol")
        
        # 找出最佳结果
        best_result = results_df.loc[results_df['best_energy'].idxmin()]
        print(f"\n最佳对接结果:")
        print(f"  蛋白质: {best_result['protein_name']}")
        print(f"  口袋排名: {best_result['pocket_rank']}")
        print(f"  结合能: {best_result['best_energy']:.2f} kcal/mol")
        print(f"  坐标: ({best_result['center_x']:.1f}, {best_result['center_y']:.1f}, {best_result['center_z']:.1f})")
    
    # 保存结果
    results_file = WORK_DIR / 'docking_analysis.csv'
    results_df.to_csv(results_file, index=False)
    print(f"\n✓ 详细结果保存: {results_file}")
    
    # 显示前10个结果
    print(f"\n前10个结果 (按结合能排序):")
    print(results_df.nsmallest(10, 'best_energy')[['protein_name', 'pocket_rank', 'best_energy', 'num_poses']])
    
    return results_df

# 解析和分析结果
if docking_results:
    parsed_results = parse_vina_results(docking_results)
    analysis_df = analyze_docking_results(parsed_results)
else:
    analysis_df = pd.DataFrame()
    print("⚠️ 没有对接结果可供分析")

## 8. 结果可视化

In [ ]:
def visualize_docking_results(results_df):
    """可视化对接结果"""
    
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. 结合能分布
        axes[0, 0].hist(results_df['best_energy'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
        axes[0, 0].set_xlabel('最佳结合能 (kcal/mol)')
        axes[0, 0].set_ylabel('频次')
        axes[0, 0].set_title('结合能分布')
        axes[0, 0].axvline(results_df['best_energy'].mean(), color='red', linestyle='--', label=f'平均值: {results_df["best_energy"].mean():.1f}')
        axes[0, 0].legend()
        
        # 2. 按蛋白质分组的最佳结合能
        if 'protein_name' in results_df.columns:
            protein_energies = results_df.groupby('protein_name')['best_energy'].min().sort_values()
            axes[0, 1].bar(range(len(protein_energies)), protein_energies.values, color='lightgreen', alpha=0.7)
            axes[0, 1].set_xlabel('蛋白质')
            axes[0, 1].set_ylabel('最佳结合能 (kcal/mol)')
            axes[0, 1].set_title('各蛋白质的最佳结合能')
            axes[0, 1].tick_params(axis='x', rotation=45)
        
        # 3. 口袋排名 vs 结合能
        if 'pocket_rank' in results_df.columns:
            rank_energies = results_df.groupby('pocket_rank')['best_energy'].mean()
            axes[1, 0].plot(rank_energies.index, rank_energies.values, 'o-', color='orange', linewidth=2, markersize=8)
            axes[1, 0].set_xlabel('口袋排名')
            axes[1, 0].set_ylabel('平均结合能 (kcal/mol)')
            axes[1, 0].set_title('口袋排名 vs 平均结合能')
            axes[1, 0].grid(True, alpha=0.3)
        
        # 4. 对接时间 vs 结合能
        if 'docking_time' in results_df.columns:
            axes[1, 1].scatter(results_df['docking_time'], results_df['best_energy'], alpha=0.6, color='purple')
            axes[1, 1].set_xlabel('对接时间 (秒)')
            axes[1, 1].set_ylabel('最佳结合能 (kcal/mol)')
            axes[1, 1].set_title('对接时间 vs 结合能')
            axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # 保存图形
        plot_file = WORK_DIR / 'docking_results.png'
        plt.savefig(plot_file, dpi=300, bbox_inches='tight')
        print(f"✓ 可视化结果保存: {plot_file}")
        
        plt.show()
        
    except ImportError:
        print("⚠️ 需要安装 matplotlib 和 seaborn 进行可视化")
        print("运行: pip install matplotlib seaborn")
    
    except Exception as e:
        print(f"❌ 可视化失败: {e}")

# 可视化结果
if not analysis_df.empty:
    visualize_docking_results(analysis_df)

## 9. 最佳结果提取

In [ ]:
def extract_best_results(results_df, top_n=5):
    """提取最佳对接结果"""
    if results_df.empty:
        return pd.DataFrame()
    
    # 按结合能排序，选择前N个
    best_results = results_df.nsmallest(top_n, 'best_energy')
    
    print(f"=== 前 {top_n} 个最佳对接结果 ===")
    
    for i, (_, result) in enumerate(best_results.iterrows(), 1):
        print(f"\n{i}. 蛋白质: {result['protein_name']}")
        print(f"   口袋排名: {result['pocket_rank']}")
        print(f"   结合能: {result['best_energy']:.2f} kcal/mol")
        print(f"   坐标: ({result['center_x']:.1f}, {result['center_y']:.1f}, {result['center_z']:.1f})")
        print(f"   输出文件: {result['output_file']}")
    
    # 保存最佳结果
    best_file = WORK_DIR / 'best_docking_results.csv'
    best_results.to_csv(best_file, index=False)
    print(f"\n✓ 最佳结果保存: {best_file}")
    
    return best_results

# 提取最佳结果
if not analysis_df.empty:
    best_results = extract_best_results(analysis_df, top_n=10)

## 10. 下一步操作

完成分子对接后，您可以：

1. **结果分析**: 
   - 分析结合模式和相互作用
   - 比较不同配体的结合亲和力
   - 研究结构-活性关系

2. **可视化验证**:
   - 使用 PyMOL 或类似工具查看对接构象
   - 分析关键相互作用 (氢键, 疏水作用等)
   - 生成结合位点图像

3. **后续实验设计**:
   - 基于对接结果设计突变实验
   - 优化配体结构
   - 规划生物化学验证实验

**结果文件位置:**
- 对接结果: `{WORK_DIR}/docking_results/`
- 结果分析: `{WORK_DIR}/docking_analysis.csv`
- 最佳结果: `{WORK_DIR}/best_docking_results.csv`
- 可视化图表: `{WORK_DIR}/docking_results.png`

**注意:**
- 结合能负值越大，结合越稳定
- 通常认为结合能 < -6.0 kcal/mol 为较好的结合
- 需要结合实验数据验证计算结果